# 환경 통합 작물 질병 진행단계 분류
## EfficientNet-B0 + CBAM | 70:15:15 분할

**실행 순서**
1. 환경 설정 & 패키지 설치
2. 설정값 (경로/하이퍼파라미터)
3. 데이터 로드 및 70:15:15 분할
4. 피처 추출 (1회만)
5. 피처 기반 MLP 학습
6. Fine-tuning 학습
7. 테스트셋 최종 평가
8. 예측 & Grad-CAM
9. 실험 A/B/C 교차 평가


## 1. 환경 설정

In [ ]:
# 패키지 설치 (최초 1회)
!pip install torch torchvision scikit-learn opencv-python-headless matplotlib tqdm -q
print('설치 완료')

In [ ]:
import sys, os, json, time
from pathlib import Path

# /workspace에 dataset.py, model.py 등 파일이 있어야 함
WORKSPACE = Path('/workspace')
sys.path.insert(0, str(WORKSPACE))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'디바이스: {DEVICE} | AMP: {"ON" if USE_AMP else "OFF"}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. 설정값 (여기서 경로/하이퍼파라미터 수정)

In [ ]:
# ── 데이터 경로 ─────────────────────────────────────────
# 각 루트 아래 images/, labels/ 가 있어야 함
DATA_ROOTS = [
    '/workspace/data/facility',   # 시설 전체
    '/workspace/data/outdoor',    # 노지 전체
]

# ── 저장 경로 ─────────────────────────────────────────────
SAVE_DIR          = '/workspace/checkpoints/exp_A'
FEATURE_DIR_TRAIN = '/workspace/data/features/train'
FEATURE_DIR_VAL   = '/workspace/data/features/val'
FEATURE_DIR_TEST  = '/workspace/data/features/test'
STATS_PATH        = '/workspace/checkpoints/dataset_stats.json'
LINEAR_SAVE_PATH  = '/workspace/checkpoints/linear_best.pth'

# ── 데이터 분할 ───────────────────────────────────────────
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
# TEST_RATIO = 0.15 (자동 계산)
SPLIT_SEED   = 42
CROP_FILTER  = {2, 7, 9, 11}  # None = 전체 작물

# ── Fine-tuning 하이퍼파라미터 ────────────────────────────
FT_EPOCHS        = 25
FT_BATCH         = 64
FT_NUM_WORKERS   = 4
FT_LR            = 1e-4
FT_WEIGHT_DECAY  = 1e-4
FT_DROPOUT       = 0.4
FT_FREEZE_EPOCHS = 3
FT_PATIENCE      = 10

# ── 피처 추출 하이퍼파라미터 ─────────────────────────────
LIN_EPOCHS  = 50
LIN_BATCH   = 256
LIN_LR      = 1e-3
LIN_DROPOUT = 0.3

for d in [SAVE_DIR, FEATURE_DIR_TRAIN, FEATURE_DIR_VAL, FEATURE_DIR_TEST]:
    Path(d).mkdir(parents=True, exist_ok=True)
print('설정 완료')

## 3. 데이터 로드 및 70:15:15 분할

In [ ]:
from dataset import (
    load_all_samples, split_samples, print_split_info,
    build_dataloaders, compute_dataset_stats,
    RISK_CLASSES, NUM_CLASSES, _MEAN, _STD,
)

# 전체 샘플 로드
all_samples = []
for root in DATA_ROOTS:
    all_samples.extend(load_all_samples(root, CROP_FILTER))
print(f'\n전체 샘플 수: {len(all_samples):,}')

In [ ]:
# 70:15:15 분할
train_s, val_s, test_s = split_samples(
    all_samples,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = SPLIT_SEED,
)
print_split_info(train_s, val_s, test_s)

In [ ]:
# 정규화 통계 (저장된 값 있으면 로드, 없으면 계산)
stats_p = Path(STATS_PATH)
if stats_p.exists():
    stats      = json.load(open(stats_p))
    MEAN, STD  = stats['mean'], stats['std']
    print(f'저장된 통계 로드: mean={MEAN}')
else:
    print('통계 계산 중 (수 분 소요)...')
    MEAN, STD = compute_dataset_stats(train_s, num_workers=FT_NUM_WORKERS)
    stats_p.parent.mkdir(parents=True, exist_ok=True)
    json.dump({'mean': MEAN, 'std': STD}, open(stats_p, 'w'))
    print(f'통계 저장: {stats_p}')

In [ ]:
# DataLoader 구성
loaders = build_dataloaders(
    train_samples        = train_s,
    val_samples          = val_s,
    test_samples         = test_s,
    batch_size           = FT_BATCH,
    num_workers          = FT_NUM_WORKERS,
    use_weighted_sampler = True,
    mean                 = MEAN,
    std                  = STD,
)
print('DataLoader 구성 완료')
for k, v in loaders.items():
    print(f'  {k:6s}: {len(v):4d} batches')

## 4. 피처 추출 (1회만 실행)
features.npy / labels.npy 이미 있으면 이 섹션 건너뜀


In [ ]:
from dataset import CropDiseaseDataset, get_transforms
from model   import EfficientNetB0CropDisease
from torch.utils.data import DataLoader as DL
from tqdm.notebook import tqdm as tqdm_nb

def extract_features(samples, save_dir, mean, std, desc='추출'):
    """샘플 리스트에서 피처 벡터(1280d)를 추출하여 npy로 저장."""
    feat_path = Path(save_dir) / 'features.npy'
    lbl_path  = Path(save_dir) / 'labels.npy'
    if feat_path.exists() and lbl_path.exists():
        print(f'{desc}: 이미 존재 → 건너뜀 ({feat_path})', flush=True)
        return

    ds = CropDiseaseDataset(samples, get_transforms('val', mean, std), desc)
    loader = DL(ds, batch_size=64, shuffle=False,
                num_workers=FT_NUM_WORKERS, pin_memory=True)

    extractor = EfficientNetB0CropDisease().to(DEVICE).eval()
    for p in extractor.parameters():
        p.requires_grad = False

    all_vecs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm_nb(loader, desc=desc):
            imgs = imgs.to(DEVICE, non_blocking=True)
            x    = extractor.features(imgs)
            x    = extractor.cbam(x)
            vec  = extractor.pool(x).flatten(1)   # (B, 1280)
            all_vecs.append(vec.cpu().numpy())
            all_labels.append(labels.numpy())

    all_vecs   = np.vstack(all_vecs)
    all_labels = np.concatenate(all_labels)
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    np.save(feat_path, all_vecs)
    np.save(lbl_path,  all_labels)
    from collections import Counter
    print(f'{desc} 완료: {all_vecs.shape} | 분포: {dict(Counter(all_labels.tolist()))}',
          flush=True)

extract_features(train_s, FEATURE_DIR_TRAIN, MEAN, STD, 'Train 피처 추출')
extract_features(val_s,   FEATURE_DIR_VAL,   MEAN, STD, 'Val 피처 추출')
extract_features(test_s,  FEATURE_DIR_TEST,  MEAN, STD, 'Test 피처 추출')

## 5. 피처 기반 MLP 학습 (Feature Extraction)

In [ ]:
from torch.utils.data import TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import f1_score
from plot_linear import LinearMetricsLogger, plot_linear_results

def load_feat(feat_dir):
    X = np.load(f'{feat_dir}/features.npy')
    y = np.load(f'{feat_dir}/labels.npy')
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long))

X_tr,  y_tr  = load_feat(FEATURE_DIR_TRAIN)
X_val, y_val = load_feat(FEATURE_DIR_VAL)
print(f'Train 피처: {X_tr.shape} | Val 피처: {X_val.shape}')

In [ ]:
# 클래스 가중치
cnt_lin = np.bincount(y_tr.numpy(), minlength=NUM_CLASSES).astype(float)
wt_lin  = cnt_lin.sum() / (NUM_CLASSES * np.maximum(cnt_lin, 1))
print('클래스 가중치:', {RISK_CLASSES[i]: f'{w:.3f}' for i, w in enumerate(wt_lin)})

crit_lin  = nn.CrossEntropyLoss(
    weight=torch.tensor(wt_lin, dtype=torch.float32).to(DEVICE)
)
tr_ldr_lin  = torch.utils.data.DataLoader(
    TensorDataset(X_tr, y_tr), batch_size=LIN_BATCH, shuffle=True)
val_ldr_lin = torch.utils.data.DataLoader(
    TensorDataset(X_val, y_val), batch_size=LIN_BATCH, shuffle=False)

classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(1280, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(inplace=True),
    nn.Dropout(LIN_DROPOUT),
    nn.Linear(512, NUM_CLASSES),
).to(DEVICE)

opt_lin = AdamW(classifier.parameters(), lr=LIN_LR, weight_decay=1e-4)
sch_lin = CosineAnnealingWarmRestarts(opt_lin, T_0=10, T_mult=1, eta_min=1e-6)
print('MLP 분류기 구성 완료')

In [ ]:
logger_lin = LinearMetricsLogger()
best_lin_f1 = 0.0
best_lin_labels, best_lin_preds = [], []

for epoch in range(1, LIN_EPOCHS + 1):
    # 학습
    classifier.train()
    tr_loss = 0.0
    for X_b, y_b in tr_ldr_lin:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        opt_lin.zero_grad()
        loss = crit_lin(classifier(X_b), y_b)
        loss.backward()
        opt_lin.step()
        tr_loss += loss.item()
    tr_loss /= len(tr_ldr_lin)

    # 검증
    classifier.eval()
    preds_ep, lbls_ep, val_loss = [], [], 0.0
    with torch.no_grad():
        for X_b, y_b in val_ldr_lin:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            out = classifier(X_b)
            val_loss += crit_lin(out, y_b).item()
            preds_ep.extend(out.argmax(1).cpu().tolist())
            lbls_ep.extend(y_b.cpu().tolist())
    val_loss /= len(val_ldr_lin)

    val_f1  = f1_score(lbls_ep, preds_ep, average='macro', zero_division=0)
    per_cls = f1_score(lbls_ep, preds_ep, average=None,    zero_division=0)
    lr_now  = opt_lin.param_groups[0]['lr']
    sch_lin.step()

    logger_lin.update(epoch, val_f1, per_cls, lr_now,
                      train_loss=tr_loss, val_loss=val_loss)

    print(f'[{epoch:3d}/{LIN_EPOCHS}] '
          f'Tr={tr_loss:.4f} Val={val_loss:.4f} F1={val_f1:.4f} | '
          f'정상:{per_cls[0]:.3f} 초기:{per_cls[1]:.3f} '
          f'중기:{per_cls[2]:.3f} 말기:{per_cls[3]:.3f}', flush=True)

    if val_f1 > best_lin_f1:
        best_lin_f1     = val_f1
        best_lin_labels = lbls_ep.copy()
        best_lin_preds  = preds_ep.copy()
        torch.save(classifier.state_dict(), LINEAR_SAVE_PATH)
        print(f'  ✔ Best 저장 ({best_lin_f1:.4f})', flush=True)

print(f'\n피처 추출 완료 — Best Val Macro-F1: {best_lin_f1:.4f}')

In [ ]:
# 피처 추출 학습 곡선 시각화
plot_linear_results(
    logger_lin,
    save_dir   = SAVE_DIR,
    all_labels = best_lin_labels,
    all_preds  = best_lin_preds,
    filename   = 'training_curves_linear.png',
)
IPImage(f'{SAVE_DIR}/training_curves_linear.png')

## 6. Fine-tuning 학습

In [ ]:
from torch.amp import GradScaler, autocast
from model       import EfficientNetB0CropDisease
from plot_results import MetricsLogger, plot_all

# 모델 생성
model_ft = EfficientNetB0CropDisease(
    num_classes  = NUM_CLASSES,
    dropout_rate = FT_DROPOUT,
).to(DEVICE)

# fc 가중치 이식 (linear_best.pth)
if Path(LINEAR_SAVE_PATH).exists():
    ls = torch.load(LINEAR_SAVE_PATH, map_location=DEVICE)
    model_ft.fc[1].weight.data.copy_(ls['1.weight'])
    model_ft.fc[1].bias.data.copy_(ls['1.bias'])
    model_ft.fc[2].weight.data.copy_(ls['2.weight'])
    model_ft.fc[2].bias.data.copy_(ls['2.bias'])
    model_ft.fc[2].running_mean.copy_(ls['2.running_mean'])
    model_ft.fc[2].running_var.copy_(ls['2.running_var'])
    model_ft.fc[2].num_batches_tracked.copy_(ls['2.num_batches_tracked'])
    model_ft.fc[5].weight.data.copy_(ls['5.weight'])
    model_ft.fc[5].bias.data.copy_(ls['5.bias'])
    print(f'fc 가중치 이식 완료: {LINEAR_SAVE_PATH}', flush=True)
else:
    print('linear_best.pth 없음 → 랜덤 초기화', flush=True)

# 손실 함수
cnt_ft = np.bincount([s['risk_label'] for s in train_s],
                      minlength=NUM_CLASSES).astype(float)
wt_ft  = cnt_ft.sum() / (NUM_CLASSES * np.maximum(cnt_ft, 1))
print('클래스 가중치:', {RISK_CLASSES[i]: f'{w:.3f}' for i, w in enumerate(wt_ft)})
crit_ft = nn.CrossEntropyLoss(
    weight=torch.tensor(wt_ft, dtype=torch.float32).to(DEVICE)
)
scaler = GradScaler(enabled=USE_AMP)
print(f'모델 준비 완료 | AMP: {"ON" if USE_AMP else "OFF"}')

In [ ]:
# 학습/평가 함수
def train_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss, preds_all, lbls_all = 0.0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        with autocast('cuda', enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        preds_all.extend(logits.argmax(1).cpu().tolist())
        lbls_all.extend(labels.cpu().tolist())
    f1 = f1_score(lbls_all, preds_all, average='macro', zero_division=0)
    return {'loss': total_loss / len(loader), 'f1': f1}

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, preds_all, lbls_all = 0.0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        preds_all.extend(logits.argmax(1).cpu().tolist())
        lbls_all.extend(labels.cpu().tolist())
    f1      = f1_score(lbls_all, preds_all, average='macro', zero_division=0)
    per_cls = f1_score(lbls_all, preds_all, average=None,    zero_division=0)
    return {'loss': total_loss/len(loader), 'f1': f1,
            'per_class_f1': per_cls, 'preds': preds_all, 'labels': lbls_all}

print('함수 정의 완료')

In [ ]:
logger_ft    = MetricsLogger()
best_ft_f1   = 0.0
patience_cnt = 0
final_labels = final_preds = None
opt_ft = sch_ft = None

print('=' * 65, flush=True)
print(f'Phase 1: {FT_FREEZE_EPOCHS}에폭 head만 학습 / Phase 2: 전체 fine-tuning', flush=True)
print('=' * 65, flush=True)

for epoch in range(1, FT_EPOCHS + 1):
    t0 = time.time()

    if epoch == 1:
        model_ft.freeze_features()
        opt_ft = AdamW(filter(lambda p: p.requires_grad, model_ft.parameters()),
                       lr=FT_LR * 5, weight_decay=FT_WEIGHT_DECAY)
        sch_ft = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt_ft, T_max=FT_FREEZE_EPOCHS, eta_min=1e-5)
        print('▶ Phase 1: features 동결', flush=True)

    elif epoch == FT_FREEZE_EPOCHS + 1:
        model_ft.unfreeze_features()
        opt_ft = AdamW([
            {'params': model_ft.features[:4].parameters(), 'lr': FT_LR * 0.1},
            {'params': model_ft.features[4:].parameters(), 'lr': FT_LR * 0.2},
            {'params': model_ft.cbam.parameters(),         'lr': FT_LR * 0.5},
            {'params': model_ft.fc.parameters(),           'lr': FT_LR},
        ], weight_decay=FT_WEIGHT_DECAY)
        sch_ft = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt_ft, T_max=FT_EPOCHS - FT_FREEZE_EPOCHS, eta_min=1e-6)
        print(f'\n▶ Phase 2 시작 (epoch {epoch})', flush=True)
        print(f'  features[:4]={FT_LR*0.1:.2e} features[4:]={FT_LR*0.2:.2e} '
              f'cbam={FT_LR*0.5:.2e} fc={FT_LR:.2e}', flush=True)

    phase = 1 if epoch <= FT_FREEZE_EPOCHS else 2
    tr    = train_epoch(model_ft, loaders['train'], opt_ft, scaler, crit_ft)
    val   = eval_epoch(model_ft,  loaders['val'],   crit_ft)
    sch_ft.step()

    lr_now  = opt_ft.param_groups[0]['lr']
    gap     = tr['f1'] - val['f1']
    elapsed = time.time() - t0
    per     = val['per_class_f1']

    logger_ft.update(epoch, tr, val, lr_now, phase)

    print(f'[Ep {epoch:3d}/{FT_EPOCHS} Ph{phase}] '
          f'Tr {tr["loss"]:.4f}/{tr["f1"]:.4f} '
          f'Val {val["loss"]:.4f}/{val["f1"]:.4f} '
          f'Gap={gap:+.3f} lr={lr_now:.2e} [{elapsed:.1f}s]', flush=True)
    print(f'  정상:{per[0]:.3f} 초기:{per[1]:.3f} 중기:{per[2]:.3f} 말기:{per[3]:.3f}',
          flush=True)

    if val['f1'] > best_ft_f1:
        best_ft_f1   = val['f1']
        patience_cnt = 0
        final_labels = val['labels']
        final_preds  = val['preds']
        torch.save({'epoch': epoch, 'state_dict': model_ft.state_dict(),
                    'scaler': scaler.state_dict(), 'best_f1': best_ft_f1},
                   f'{SAVE_DIR}/best_model.pth')
        print(f'  ✔ Best 저장 (Val F1: {best_ft_f1:.4f})', flush=True)
    else:
        patience_cnt += 1

    if patience_cnt >= FT_PATIENCE:
        print(f'\n조기 종료: {FT_PATIENCE}에폭 개선 없음.', flush=True)
        break

print(f'\n학습 완료 — Best Val Macro-F1: {best_ft_f1:.4f}', flush=True)

In [ ]:
# Fine-tuning 학습 곡선 시각화
plot_all(logger_ft, save_dir=SAVE_DIR,
         all_labels=final_labels, all_preds=final_preds)
IPImage(f'{SAVE_DIR}/training_curves.png')

## 7. 테스트셋 최종 평가

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# Best 모델 로드
ckpt_best = torch.load(f'{SAVE_DIR}/best_model.pth',
                        map_location=DEVICE, weights_only=False)
model_ft.load_state_dict(ckpt_best['state_dict'])
print(f'Best 모델 로드 (epoch={ckpt_best["epoch"]}, '
      f'best_f1={ckpt_best["best_f1"]:.4f})')

# 테스트 평가
test_res = eval_epoch(model_ft, loaders['test'], crit_ft)
per = test_res['per_class_f1']
print(f'\n[테스트셋] Macro-F1: {test_res["f1"]:.4f}')
print(f'  정상:{per[0]:.3f} 초기:{per[1]:.3f} 중기:{per[2]:.3f} 말기:{per[3]:.3f}')
print()
print(classification_report(
    test_res['labels'], test_res['preds'],
    target_names=['Norm.','Early','Mid.','Term.'], zero_division=0
))

In [ ]:
# Confusion Matrix
cm  = confusion_matrix(test_res['labels'], test_res['preds'])
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=['Norm.','Early','Mid.','Term.']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Test Confusion Matrix | Macro-F1: {test_res["f1"]:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/test_confusion_matrix.png', dpi=150)
plt.show()

## 8. 예측 & Grad-CAM

In [ ]:
import cv2
from PIL import Image as PILImage
from torchvision import transforms as T

_preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class GradCAM:
    def __init__(self, model):
        self.model = model
        self.activations = self.gradients = None
        model.features[-1].register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        model.features[-1].register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, tensor, target=None):
        self.model.eval()
        tensor = tensor.clone().requires_grad_(True)
        logits = self.model(tensor)
        if target is None:
            target = logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, target].backward()
        w   = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.activations).sum(1).squeeze()).cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target

def predict_and_show(img_path, model, show_gradcam=True):
    img_pil = PILImage.open(img_path).convert('RGB')
    tensor  = _preprocess(img_pil).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1).squeeze().cpu().numpy()
    pred = int(probs.argmax())

    print(f'예측: {RISK_CLASSES[pred]} ({probs[pred]:.4f})')
    for k, name in RISK_CLASSES.items():
        print(f'  {name:4s}: {probs[k]:.4f} {"█" * int(probs[k]*30)}')

    if show_gradcam:
        gcam = GradCAM(model)
        cam, _ = gcam.generate(tensor, pred)
        img_cv  = cv2.imread(str(img_path))
        img_cv  = cv2.resize(img_cv, (224, 224))
        heat    = cv2.applyColorMap(
            np.uint8(255 * cv2.resize(cam, (224, 224))), cv2.COLORMAP_JET)
        overlay = cv2.addWeighted(img_cv, 0.5, heat, 0.5, 0)

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
        axes[0].set_title('원본'); axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f'Grad-CAM: {RISK_CLASSES[pred]}')
        axes[1].axis('off')
        plt.tight_layout(); plt.show()

# 사용 예시:
# predict_and_show('/workspace/data/facility/images/sample.jpg', model_ft)

## 9. 실험 A/B/C 교차 평가
*(실험 B, C 학습 완료 후 실행)*


In [ ]:
# 환경별 테스트셋 분리
fac_test = [s for s in test_s if s['env'] == '시설']
out_test  = [s for s in test_s if s['env'] == '노지']
print(f'시설 테스트: {len(fac_test):,}개 | 노지 테스트: {len(out_test):,}개')

from dataset import CropDiseaseDataset, get_transforms

def make_test_loader(samples):
    ds = CropDiseaseDataset(samples, get_transforms('val', MEAN, STD), 'test')
    return torch.utils.data.DataLoader(
        ds, batch_size=64, shuffle=False, num_workers=FT_NUM_WORKERS)

ldr_fac  = make_test_loader(fac_test)
ldr_out  = make_test_loader(out_test)
ldr_all  = make_test_loader(test_s)

def eval_print(model, loader, label):
    r   = eval_epoch(model, loader, crit_ft)
    per = r['per_class_f1']
    print(f'  {label:25s} F1={r["f1"]:.4f} | '
          f'정상:{per[0]:.3f} 초기:{per[1]:.3f} 중기:{per[2]:.3f} 말기:{per[3]:.3f}',
          flush=True)
    return r

print('\n[실험 A — 통합 모델]')
eval_print(model_ft, ldr_fac, '× 시설 테스트')
eval_print(model_ft, ldr_out, '× 노지 테스트')
eval_print(model_ft, ldr_all, '× 통합 테스트')

# 실험 B / C 체크포인트 준비 후:
# model_B = EfficientNetB0CropDisease(NUM_CLASSES, FT_DROPOUT).to(DEVICE)
# ckpt_B  = torch.load('.../exp_B/best_model.pth', map_location=DEVICE, weights_only=False)
# model_B.load_state_dict(ckpt_B['state_dict'])
# crit_B  = nn.CrossEntropyLoss(weight=...)
# print('\n[실험 B — 시설 단독]')
# eval_print(model_B, ldr_fac, '× 시설 테스트')
# eval_print(model_B, ldr_out, '× 노지 테스트 (교차)')